# NB-15 — clean WER for every replicate checkpoint

**~4 GPU-minutes per checkpoint. No test set, no FLEURS, no attack.**

## Why this exists

Clean WER on the frozen 100-utterance attack set is the **ANCOVA covariate**. A replicate
produced by NB-14 cannot enter the pooled fit without it, and it is also what makes the
variance claim interesting: the paper's headline rests on two models that agree to 0.0104 WER
and still differ by 34 pp in attack success. Each new replicate needs the same two numbers.

It is cheap and it gates everything downstream, so run it before spending 4.6 GPU-h on an
attack. If a replicate's clean WER lands far from its published sibling, you have a different
condition rather than a second draw, and you want to know that **before** you attack it.

## What it reuses unchanged

`transcribe`, `_edit` and `score` are byte-identical to NB-02c. `score` is corpus-level WER
and CER — total edits over total reference length, jiwer's own aggregation — implemented inline
because **`jiwer` is not preinstalled on Kaggle** and the pip cell cannot reach the index with
internet off. It is verified against jiwer 4.x to machine precision.

## Attach

Every checkpoint you want scored, plus `bn-asr-splits`, `bengaliai-speech`, and any one
`zoayriaabedin/eps50-*-n100` (for the frozen 100-utterance list and its sha256).

Each NB-14 output is ~5 GB (measured on `ckpt-xlsr-1h-s1338`: `final/` plus one spare
`ckpt/checkpoint-6000`), so ten checkpoints is ~50 GB of read-only mount. That does not touch
the 20 GB working-disk budget, so try all ten in one commit first. Only if Kaggle refuses the
attachment should you split: run this notebook twice with different `CKPT_DATASETS` and merge
the CSVs — the frozen-target sha256 guard makes the two runs comparable by construction.

Nothing here reads `ckpt/`; only `final/`.

GPU T4 x2. Internet off is fine.


## 1 · Config — list every checkpoint to score

In [ ]:
# Every checkpoint to score, in any order. A name that is not attached is REPORTED
# and skipped, not raised: a partial table is useful and a 20-minute run that dies
# on the last missing mount is not.
#
# The two published 1 h runs are included on purpose. They are the pair behind the
# paper's 34 pp result (0.6355 and 0.6251), so re-measuring them here proves this
# notebook reproduces the numbers the paper already reports before you trust it on
# a checkpoint nobody has scored.
CKPT_DATASETS = [
    "ckpt-xlsr-1h",            # published, tf 4.44.2, best step 2500  -> expect 0.6355
    "ckpt-xlsr-1h-traj",       # second run, tf 4.44.2, best step 5500 -> expect 0.6251
    "ckpt-xlsr-1h-s1338",      # replicate, current tf
    "ckpt-xlsr-1h-s1339",      # replicate, current tf  <- the clean 1 h pair is these two
    "ckpt-xlsr-5h",            # published, tf 4.44.2
    "ckpt-xlsr-5h-s1338",
    "ckpt-xlsr-5h-s1339",      # <- the clean 5 h pair
    "ckpt-xlsr-25h",           # published, tf 4.44.2
    "ckpt-xlsr-25h-s1338",
    "ckpt-xlsr-25h-s1339",     # <- the clean 25 h pair
]
CHECKPOINT   = "final"
MAX_DUR      = 10.0
EVAL_BS      = 8
SEED         = 1337          # inference seed; greedy CTC, so this is cosmetic

# Published figures, all 100 utterances, for the agreement check at the end.
# Source: RUNLOG 2026-10-07 (NB-02b / NB-02c).
KNOWN = {"ckpt-xlsr-1h": 0.6355, "ckpt-xlsr-1h-traj": 0.6251}

import os, sys, json, time, re, glob, random, unicodedata, hashlib
import numpy as np, pandas as pd, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, DEVICE = "/kaggle/working", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"
assert len(set(CKPT_DATASETS)) == len(CKPT_DATASETS), "duplicate name in CKPT_DATASETS"
print(f"NB-15 | {len(CKPT_DATASETS)} checkpoints requested | clean WER on the 100-utt attack set")

## 2 · Pins

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# NOTE: that cell also installed `jiwer`, which is NOT preinstalled on Kaggle.
# WER/CER are therefore implemented inline below, verified against jiwer to
# machine precision. transformers, librosa and numpy ARE preinstalled.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")


## 3 · Inputs

In [ ]:
def list_inputs():
    out = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))

INPUTS_LIST = list_inputs()

def find_input(pattern, what, required=True):
    hits = [p for p in INPUTS_LIST if re.search(pattern, os.path.basename(p), re.I)]
    if not hits:
        if not required: return None
        print("Visible under /kaggle/input:"); [print("   ", p) for p in INPUTS_LIST]
        raise AssertionError(f"Could not find {what} (regex {pattern})")
    return max(hits, key=lambda p: (p.count("/"), len(p)))

# NB-02b's model-discovery loop is deliberately NOT carried over: it iterates
# RUNGS over ckpt-xlsr-{r}h, and this notebook resolves exactly one
# checkpoint, by name, in the next cell.


## 4 · Resolve the checkpoint and the frozen 100

In [ ]:
# ---------------------------------------------------------------------------
# Resolve every requested checkpoint.
#
# The regex is anchored with `$` for a reason: without it "ckpt-xlsr-1h" also
# matches "ckpt-xlsr-1h-traj" and "ckpt-xlsr-1h-s1338", and the published rung
# would be scored three times under three different labels. With it, each name
# matches only itself.
# ---------------------------------------------------------------------------
MODELS, MISSING = [], []
for ds in CKPT_DATASETS:
    cands = [p for p in INPUTS_LIST
             if re.search(rf"{re.escape(ds)}$", os.path.basename(p))]
    if len(cands) != 1:
        MISSING.append((ds, [os.path.basename(c) for c in cands]))
        continue
    ck = cands[0]
    md = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    if not os.path.exists(f"{md}/config.json"):
        MISSING.append((ds, ["mounted but no config.json under it or its final/"]))
        continue
    MODELS.append(dict(name=ds, model_dir=md))

print(f"resolved {len(MODELS)}/{len(CKPT_DATASETS)} checkpoints:")
for m in MODELS:
    print(f"   {m['name']:24s} -> {m['model_dir']}")
if MISSING:
    print(f"\nNOT SCORED ({len(MISSING)}) -- attach these and re-commit to add them:")
    for ds, hits in MISSING:
        print(f"   {ds:24s} matches: {hits}")
assert MODELS, ("none of the requested checkpoints is attached.\n"
                f"Visible: {[os.path.basename(p) for p in INPUTS_LIST]}")

SPLITS = next((p for p in INPUTS_LIST if "splits" in p.lower()), None)
assert SPLITS, "attach bn-asr-splits"
TGT = next((f"{p}/task_b_targets_n100.csv" for p in INPUTS_LIST
            if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT, "attach one eps50-*-n100 dataset for the frozen 100-utterance list"
SHA = hashlib.sha256(open(TGT, "rb").read()).hexdigest()
assert SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "not the frozen target file the attacks used; the WER would not match the set")
print("targets:", TGT, "\nsha256 :", SHA)

ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

def fix_paths(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0): return df
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    c = [d for pat in (f"/kaggle/input/*/{stem}", f"/kaggle/input/*/*/{stem}",
                       f"/kaggle/input/*/*/*/{stem}")
         for d in glob.glob(pat) if os.path.isdir(d)]
    assert c, f"audio dir '{stem}' not found -- attach bengaliai-speech"
    df = df.copy(); df["path"] = c[0] + "/" + df["utt_id"].astype(str) + ext
    return df

tg = pd.read_csv(TGT)
assert len(tg) == 100 and tg.utt_id.nunique() == 100
_full = pd.read_csv(f"{SPLITS}/test.csv")
attack = tg[["utt_id"]].merge(_full, on="utt_id", how="left")
assert attack["path"].notna().all(), "some of the 100 are absent from test.csv"
assert (attack.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
attack = fix_paths(attack)
N25 = 25   # manifest order: the first 25 are the published RQ1 spine
print(f"attack set: {len(attack)} utterances, first {N25} = the RQ1 spine")

## 5 · Transcribe and score every checkpoint

In [ ]:
def load_wavs(paths):
    out = []
    for p in paths:
        try:
            a, _ = librosa.load(p, sr=16000, mono=True, duration=MAX_DUR)
            a = np.asarray(a, np.float32)
            pk = np.abs(a).max()
            out.append(a / pk if pk > 1.0 else a)
        except Exception:
            out.append(np.zeros(16000, np.float32))
    return out


@torch.no_grad()
def transcribe(model, processor, wavs, bs=EVAL_BS):
    """Length-sorted batched greedy CTC decode. Order is restored at the end."""
    order = np.argsort([len(w) for w in wavs])
    hyps = [None] * len(wavs)
    do_norm = bool(getattr(processor.feature_extractor, "do_normalize", True))
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]
        chunk = [wavs[j] for j in idx]
        T = max(len(w) for w in chunk)
        x = torch.zeros(len(chunk), T, device=DEVICE)
        m = torch.zeros(len(chunk), T, device=DEVICE)
        for k, w in enumerate(chunk):
            x[k, :len(w)] = torch.from_numpy(w).to(DEVICE); m[k, :len(w)] = 1
        if do_norm:
            n = m.sum(-1, keepdim=True).clamp(min=1)
            mu = (x*m).sum(-1, keepdim=True)/n
            var = (((x-mu)*m)**2).sum(-1, keepdim=True)/n
            x = ((x-mu)/torch.sqrt(var+1e-7))*m
        ids = torch.argmax(model(input_values=x, attention_mask=m.long()).logits, -1)
        for k, j in enumerate(idx):
            hyps[j] = norm_bn(processor.batch_decode(ids[k:k+1])[0])
    return hyps


def _edit(r, h):
    """Levenshtein on token lists."""
    if not r: return len(h)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = d[j]
            d[j] = min(d[j] + 1, d[j-1] + 1, prev + (r[i-1] != h[j-1]))
            prev = cur
    return d[len(h)]

def score(refs, hyps):
    """CORPUS-level WER and CER: total edits / total reference length.

    Implemented inline rather than with jiwer, because `jiwer` is NOT preinstalled
    on Kaggle -- it was supplied by the pip cell, which cannot run with internet
    off. Verified against jiwer 4.x to machine precision on light/heavy/large
    corpora and on both degenerate cases (empty hypothesis -> 1.0, over-long
    hypothesis -> 1.5, i.e. WER may exceed 1.0).

    Corpus-level, NOT the mean of per-utterance rates. jiwer.wer(list, list)
    aggregates, so every published figure in this project is an aggregate, and a
    per-utterance mean would silently differ.
    """
    pairs = [(norm_bn(r), h) for r, h in zip(refs, hyps) if str(r).strip()]
    if not pairs: return 1.0, 1.0
    R = [p[0] for p in pairs]; H = [p[1] for p in pairs]
    wn = sum(_edit(r.split(), h.split()) for r, h in zip(R, H))
    wd = sum(len(r.split()) for r in R)
    cn = sum(_edit(list(r), list(h)) for r, h in zip(R, H))
    cd = sum(len(r) for r in R)
    return (wn / wd if wd else 1.0), (cn / cd if cd else 1.0)


print("loading audio...")
t0 = time.time()
ATTACK_WAVS = load_wavs(attack["path"].tolist())
print(f"  {len(ATTACK_WAVS)} clips in {time.time()-t0:.0f}s")

In [ ]:
# ---------------------------------------------------------------------------
# One model at a time. Each is loaded, scored, and FREED before the next is
# loaded: eight XLS-R-300m models do not coexist in 16 GB, and a run that dies
# of OOM on checkpoint six has wasted five scorings.
#
# The audio is decoded once, above, and reused -- it is the same 100 clips for
# every model, so re-decoding per model would multiply the runtime by eight for
# no change in result.
# ---------------------------------------------------------------------------
import gc, transformers
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

refs = list(attack["text"])
rows = []

for mi, m in enumerate(MODELS, 1):
    t0 = time.time()
    print(f"[{mi}/{len(MODELS)}] {m['name']}", flush=True)
    proc = Wav2Vec2Processor.from_pretrained(m["model_dir"])
    mdl  = Wav2Vec2ForCTC.from_pretrained(m["model_dir"]).float().to(DEVICE).eval()
    mdl.config.apply_spec_augment = False
    hyp = transcribe(mdl, proc, ATTACK_WAVS)
    w100, c100 = score(refs, hyp)
    w25,  c25  = score(refs[:N25], hyp[:N25])

    # Parse rung and seed out of the dataset name so the variance analysis can
    # group by them without a second lookup table. `-traj` is the paper's second
    # 1 h run: same seed 1337, different training run.
    rung = int(re.search(r"ckpt-xlsr-(\d+)h", m["name"]).group(1))
    sm   = re.search(r"-s(\d+)$", m["name"])
    seed = int(sm.group(1)) if sm else 1337
    run_label = m["name"].replace("ckpt-xlsr-", "")

    rows.append(dict(model=m["name"], run_label=run_label, rung_hours=rung, seed=seed,
                     checkpoint=CHECKPOINT, model_dir=m["model_dir"], n=len(hyp),
                     wer_attack100=w100, cer_attack100=c100,
                     wer_attack25=w25,  cer_attack25=c25,
                     secs=round(time.time() - t0, 1)))
    print(f"      WER100 {w100:.4f}  CER100 {c100:.4f}  WER25 {w25:.4f}  "
          f"({time.time()-t0:.0f}s)", flush=True)

    del mdl, proc
    gc.collect(); torch.cuda.empty_cache()

out = pd.DataFrame(rows).sort_values(["rung_hours", "run_label"]).reset_index(drop=True)
out.to_csv(f"{WORK}/clean_wer_replicates.csv", index=False)

print("\n" + "=" * 74)
print("CLEAN WER ON THE 100-UTTERANCE ATTACK SET  --  the ANCOVA covariate")
print("=" * 74)
print(out[["model", "rung_hours", "seed", "wer_attack100", "cer_attack100",
           "wer_attack25"]].to_string(index=False))

# ---- does this notebook reproduce the two figures the paper already reports? --
print("\n" + "-" * 74)
print("AGREEMENT CHECK against the published figures")
print("-" * 74)
bad = 0
for name, expect in KNOWN.items():
    got = out.loc[out.model == name, "wer_attack100"]
    if got.empty:
        print(f"  {name:24s} not attached -- cannot check"); continue
    g = float(got.iloc[0]); d = g - expect
    flag = "OK" if abs(d) < 5e-4 else "MISMATCH"
    bad += flag == "MISMATCH"
    print(f"  {name:24s} published {expect:.4f}  measured {g:.4f}  ({d:+.4f})  {flag}")
if bad:
    print("\n  *** A MISMATCH means this run is not measuring what the paper measured.")
    print("      Check the target sha256, the audio root and the transformers version")
    print("      BEFORE trusting any new number in the table above. ***")
else:
    print("\n  Reproduces the published figures. New numbers above are trustworthy.")

# ---- between-run spread in clean WER, per rung -----------------------------
print("\n" + "-" * 74)
print("BETWEEN-RUN SPREAD IN CLEAN WER, per rung")
print("-" * 74)
print("  This is NOT the headline. Clean WER is expected to be stable across runs;")
print("  the paper's point is that attack success is not, at the same clean WER.")
print("  A rung whose runs differ a lot in clean WER has a weaker variance claim,")
print("  because then the runs are not accuracy-matched. Check that here first.")
for rung, g in out.groupby("rung_hours"):
    if len(g) < 2:
        print(f"  {rung:3d} h: 1 run only -- no spread yet"); continue
    v = g.wer_attack100.values
    print(f"  {rung:3d} h: {len(g)} runs  WER {v.min():.4f}-{v.max():.4f}  "
          f"range {v.max()-v.min():.4f}  sd {v.std(ddof=1):.4f}")

json.dump(dict(notebook="NB-15-clean-wer-replicates", n_models=len(MODELS),
               requested=CKPT_DATASETS, missing=[d for d, _ in MISSING],
               rows=rows, targets_sha256=SHA, n=len(refs), seed=SEED,
               torch=torch.__version__, transformers=transformers.__version__,
               gpu=torch.cuda.get_device_name(0),
               timestamp=time.strftime("%Y-%m-%d %H:%M:%S")),
          open(f"{WORK}/_provenance.json", "w"), indent=2, default=str)
json.dump({"title": "clean-wer-replicates", "id": "YOURUSERNAME/clean-wer-replicates",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print("\nFILES WRITTEN")
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nPublish as 'clean-wer-replicates'. CHECK THE LIST ABOVE FIRST: publishing")
print("replaces the dataset and deletes whatever this run did not regenerate.")

---

Send me `wer_attack100`. It goes straight into the pooled ANCOVA as the seventh condition's
covariate, and it decides whether the central interval excludes zero or not.

---

## Gate before NB-16

**The agreement check must pass.** If `ckpt-xlsr-1h` does not come back at 0.6355 and
`ckpt-xlsr-1h-traj` at 0.6251, stop: something about the eval set, the audio root or the
decode differs from the published runs, and every new number is suspect.

Then, for each replicate, compare its clean WER against its published sibling:

- **Within ~0.02** — a genuine replicate. Attack it (NB-16).
- **Further than that** — report it, attack it anyway, and say so in the paper. A replicate
  that drifts in accuracy is still a draw from the recipe; it just means the between-run
  variance shows up in accuracy as well as in attackability, which is a finding rather than a
  problem.

Do not drop a replicate because its WER is inconvenient.
